In [16]:
import pandas as pd
import os
import datasets

In [36]:
# Path to .csv file containing the dataset
csv_file = '/projects/iris/rferreir/GeoBenchmark/data/GeoQuery/t.csv'

# Path to the directory containing the HF datasets 
output_path = "/projects/iris/rferreir/hub_datasets"

In [37]:
data = pd.read_csv(csv_file)

In [38]:
train = data[data['split'] == 'train'].drop(columns=['split'])

In [39]:
test = data[data['split'] == 'test'].drop(columns=['split'])

In [40]:
val = data[data['split'] == 'dev'].drop(columns=['split'])

In [41]:
d1 = {}

In [42]:
columns_to_keep = ['question', 'answer']
for cat in ["place", "regression"]:
    test_split = test[test["type"] == cat][columns_to_keep]
    
    train_split = train[train["type"] == cat][columns_to_keep]
    
    val_split = val[val["type"] == cat][columns_to_keep]

    d1[cat] = datasets.DatasetDict({
        "train": datasets.Dataset.from_pandas(train_split),
        "validation": datasets.Dataset.from_pandas(val_split),
        "test": datasets.Dataset.from_pandas(test_split)
    })

    print(d1[cat])
    print(d1[cat]['train'][0])

    d1[cat].save_to_disk(os.path.join(output_path, f"GeoQuery_{cat}"))

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', '__index_level_0__'],
        num_rows: 346
    })
    validation: Dataset({
        features: ['question', 'answer', '__index_level_0__'],
        num_rows: 33
    })
    test: Dataset({
        features: ['question', 'answer', '__index_level_0__'],
        num_rows: 184
    })
})
{'question': 'What state has highest elevation ?', 'answer': "['pennsylvania']", '__index_level_0__': 2}


Saving the dataset (1/1 shards): 100%|██████████| 184/184 [00:00<00:00, 30425.86 examples/s]


DatasetDict({
    train: Dataset({
        features: ['question', 'answer', '__index_level_0__'],
        num_rows: 182
    })
    validation: Dataset({
        features: ['question', 'answer', '__index_level_0__'],
        num_rows: 17
    })
    test: Dataset({
        features: ['question', 'answer', '__index_level_0__'],
        num_rows: 89
    })
})
{'question': 'What is the elevation of the highest point in the usa ?', 'answer': '[979.0]', '__index_level_0__': 0}


Saving the dataset (1/1 shards): 100%|██████████| 89/89 [00:00<00:00, 15962.24 examples/s]


In [44]:
import hashlib
import json

for cat in ["place", "regression"]:
    d2 = datasets.load_dataset("rfr2003/Geo_Benchmark", f"GeoQuery_{cat}")
    # We check that the content of the dataset is the same
    def content_hash(ds):
        h = hashlib.sha256()
        for ex in ds:
            h.update(json.dumps(ex, sort_keys=True).encode())
        return h.hexdigest()
    
    for split in ['train', 'test', 'validation']:
        assert content_hash(d1[cat][split]) == content_hash(d2[split])

Generating test split: 100%|██████████| 89/89 [00:00<00:00, 16060.45 examples/s]
